<a href="https://colab.research.google.com/github/julschleinitz/ai4chemistry-bootcamp/blob/main/tutorials/molecular-representations.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Tutorial 1 — Molecular Representations
## AI for Chemical Sciences Bootcamp · Caltech, August 2026

**Instructor:** Jules Schleinitz  
**Estimated time:** 60 min

---

### What you will learn

How you encode a molecule determines what a model can learn. In this tutorial we build and compare several representations of the same molecules — from your own hand-designed features to RDKit descriptors, fingerprints, and graphs — and explore them visually on the **ESOL aqueous solubility** dataset.

| Step | Section | What you'll do |
|------|---------|-----------------|
| 1 | Dataset | Load and explore the ESOL solubility data |
| 2 | Visualization toolkit | Build one reusable tool that projects any representation to 2D and plots it interactively |
| 3 | Design your own representation | Hypothesize which molecular properties drive solubility and code them up yourself |
| 4–6 | Representations | Build RDKit descriptors (+ charge/pKa), Morgan fingerprints, and a molecular graph |
| 7 | Preview | Fit a simple Ridge regression model — full modeling (Random Forests, neural networks, GNNs) is covered in later lectures |

By the end you will understand how different representations encode the same molecule, be able to visualize their chemical space interactively, and see a first, minimal example of turning a representation into a prediction.

---
## 0. Setup

In [ ]:
# Install dependencies (Colab only — comment out if running locally)
!pip install rdkit torch torch-geometric deepchem pandas scikit-learn matplotlib seaborn bokeh umap-learn transformers -q

In [ ]:
# xtb (Section 8) is a conda-forge binary, not a pip package — Colab has no conda by default.
# Uncomment and run this cell ONLY on Colab; it installs conda via condacolab and RESTARTS
# the Python runtime, so re-run the notebook from the top afterward (don't continue cell-by-cell
# in the same runtime session). Skip this cell entirely if running locally with the conda env
# from environment.yml, which already includes xtb.

# !pip install -q condacolab
# import condacolab
# condacolab.install()

In [ ]:
# Run this cell only after condacolab.install() above has restarted the runtime (Colab only).
# !mamba install -y -c conda-forge xtb

In [ ]:
import base64
import io
import subprocess
import tempfile
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# RDKit
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, AllChem, rdMolDescriptors, rdFingerprintGenerator
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit import DataStructs

# Scikit-learn
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# UMAP (another dimensionality-reduction method, alongside PCA/t-SNE)
import umap

# PyTorch / PyTorch Geometric (used to build the graph representation later)
import torch
from torch_geometric.data import Data

# HuggingFace transformers (ChemBERTa foundation model, Section 9)
from transformers import AutoTokenizer, AutoModel

# Bokeh (interactive chemical-space plots)
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource, HoverTool, LinearColorMapper, ColorBar
from bokeh.palettes import Viridis256

output_notebook()

print("All imports OK")

In [ ]:
# Data files this notebook reads (pKa and xtb-solvation CSVs) live in tutorials/data/ alongside
# the notebook. That's already true when running locally from a repo checkout — but Colab, when
# opened via the "Open in Colab" badge, only has the notebook itself, none of the repo's other
# files. Fetch them here so the `data/...` paths used below work either way.
import os
import urllib.request

DATA_FILES = ['esol_pka.csv', 'esol_xtb_solvation.csv']
DATA_BASE_URL = 'https://raw.githubusercontent.com/julschleinitz/ai4chemistry-bootcamp/main/tutorials/data/'

os.makedirs('data', exist_ok=True)
for fname in DATA_FILES:
    fpath = f'data/{fname}'
    if not os.path.exists(fpath):
        try:
            urllib.request.urlretrieve(DATA_BASE_URL + fname, fpath)
            print(f"Downloaded {fname}")
        except Exception as e:
            raise RuntimeError(
                f"Could not fetch {fname} from {DATA_BASE_URL}{fname} ({e}). "
                "If running locally, make sure tutorials/data/ contains this file."
            ) from e
    else:
        print(f"{fname} already present")

---
## 1. Dataset — ESOL aqueous solubility

The **ESOL** dataset (Delaney, 2004) contains measured aqueous solubility (log mol/L) for 1,128 small organic molecules. It is one of the most widely used benchmarks for molecular property prediction.

**Target:** `measured log(solubility:mol/L)` — a continuous value, so this is a regression task.

> **Why solubility?** It is directly relevant to drug formulation and agrochemistry, and it depends on a mix of electronic, steric, and hydrogen-bonding features — a good test for different representations.

In [ ]:
# Load ESOL directly from DeepChem's MoleculeNet — the full, unsplit dataset.
# (No train/val/test split at load time: every representation in this notebook is built
# on the full dataset; a split is only introduced later, right where a model needs one.)
import deepchem as dc

tasks, datasets, transformers = dc.molnet.load_delaney(featurizer='Raw', splitter=None)
full_ds = datasets[0]

# Convert to a flat DataFrame for easier manipulation
# featurizer='Raw' puts RDKit Mol objects in ds.X, not SMILES strings —
# ds.ids holds the original SMILES regardless of featurizer, so use that.
def dataset_to_df(ds):
    smiles = list(ds.ids)
    y = ds.y.flatten()
    return pd.DataFrame({'smiles': smiles, 'logS': y})

df_all = dataset_to_df(full_ds)
print(f"Dataset ESOL Delaney contains {len(df_all)} molecules.")
df_all.head()

## 2. RDKit Setup

### A Quick Tour of RDKit — Molecules, Visualization, and Descriptors

Everything in this notebook starts from a **SMILES string** (e.g. `CC(=O)O` for acetic acid) and turns it into an RDKit `Mol` object — RDKit's central data structure for a molecule (atoms, bonds, and all their properties). Almost every RDKit function takes a `Mol`, not a SMILES string directly, so `Chem.MolFromSmiles(...)` is the first line of nearly everything we do below.

Once you have a `Mol`, you can:
- **Inspect it**: iterate over `mol.GetAtoms()` / `mol.GetBonds()`, or compute properties like molecular formula (`rdMolDescriptors.CalcMolFormula`).
- **Draw it**: `Draw.MolToImage(mol)` for a single molecule, `Draw.MolsToGridImage([...])` for several at once (we'll use this a few cells down).
- **Compute descriptors on it**: the `rdkit.Chem.Descriptors` module implements ~200 physicochemical descriptors (molecular weight, LogP, TPSA, H-bond counts, ring counts, ...) as plain functions that take a `Mol` and return a number. This is the module behind the "Compare with an Expert-Defined Representation" section later in this notebook.

**Finding things in RDKit's documentation:** RDKit's API is large and not always intuitively organized, so it helps to know a few ways to search it:
- The [Getting Started in Python](https://www.rdkit.org/docs/GettingStartedInPython.html) page and the [RDKit Cookbook](https://www.rdkit.org/docs/Cookbook.html) cover most common tasks (reading/writing molecules, drawing, substructure search, descriptors, fingerprints) with runnable examples.
- The full [Python API reference](https://www.rdkit.org/docs/api-docs.html) documents every module — but in practice, `dir(module)` and `help(function)` directly in Python/Jupyter (e.g. `help(Descriptors.MolLogP)`) is often the fastest way to see what's available and what it returns.
- To see *every* available descriptor at once, use `Descriptors._descList` (a list of `(name, function)` pairs) or `Descriptors.CalcMolDescriptors(mol)` (computes all of them for one molecule and returns a dict) — demonstrated below.
- If you know roughly what you want but not the exact function name, searching "rdkit &lt;concept&gt;" (e.g. "rdkit number of rings", "rdkit remove stereochemistry") usually surfaces the right function faster than browsing the reference alphabetically.
- [Greg Landrum's RDKit blog](https://greglandrum.github.io/rdkit-blog/) (Greg is one of RDKit's core developers) is full of worked examples and tips that often go beyond what the official docs cover — worth searching before assuming something isn't possible in RDKit.

In [ ]:
# --- SMILES -> Mol -> inspect -> draw ---
example_smiles = df_all['smiles'].iloc[0]
mol = Chem.MolFromSmiles(example_smiles)

print(f"SMILES:            {example_smiles}")
print(f"Molecular formula: {rdMolDescriptors.CalcMolFormula(mol)}")
print(f"Number of atoms:   {mol.GetNumAtoms()} (heavy atoms only — RDKit hides Hs by default)")
print(f"Number of bonds:   {mol.GetNumBonds()}")
print(f"Atom symbols:      {[atom.GetSymbol() for atom in mol.GetAtoms()]}")

Draw.MolToImage(mol, size=(300, 200))

The following cell illustrates how RdKit can be used to directly compute descriptors for a molecule.

RDKit contains a large number of predefined descriptors, which can be accessed via the `rdkit.Chem.Descriptors` module. The following code snippet shows how to compute all available descriptors for a single molecule.

RDKit also enables the computation of molecular fingerprints (ECFP, MACCS, etc.) and molecular graphs (for use in graph neural networks). These representations will be explored in later sections of this notebook.

Finally RDKit can be used to design custom molecular representations.

In [ ]:
# --- The Descriptors module: ~200 physicochemical descriptors, one function each ---
from rdkit.Chem import Descriptors
print(f"Descriptors.MolWt(mol)   = {Descriptors.MolWt(mol):.2f}")
print(f"Descriptors.MolLogP(mol) = {Descriptors.MolLogP(mol):.2f}")
print(f"Descriptors.TPSA(mol)    = {Descriptors.TPSA(mol):.2f}")

# `_descList` holds every (name, function) pair RDKit ships — useful for browsing what's available
print(f"\nRDKit ships {len(Descriptors._descList)} descriptor functions in total, e.g.:")
for name, fn in Descriptors._descList[:5]:
    print(f"  {name:20s} -> {fn(mol):.3f}")

# Or compute all of them at once for a given molecule:
all_descriptors = Descriptors.CalcMolDescriptors(mol)
print(f"\nCalcMolDescriptors(mol) returns a dict with {len(all_descriptors)} entries, e.g.:")
{k: round(v, 3) for k, v in list(all_descriptors.items())[:5]}

In [ ]:
# note for documentation on plotting we recommend these libraries to begin with: https://seaborn.pydata.org/ and https://matplotlib.org/stable/gallery/index.html
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Solubility distribution
axes[0].hist(df_all['logS'], bins=40, color='#5DDAB4', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('log(S) [mol/L]')
axes[0].set_ylabel('Count')
axes[0].set_title('ESOL — solubility distribution')

# Molecular weight vs solubility
mws = []
for smi in df_all['smiles']:
    mol = Chem.MolFromSmiles(smi)
    mws.append(Descriptors.MolWt(mol) if mol else np.nan)
axes[1].scatter(mws, df_all['logS'], alpha=0.4, s=12, c='#5DDAB4')
axes[1].set_xlabel('Molecular weight (Da)')
axes[1].set_ylabel('log(S) [mol/L]')
axes[1].set_title('MW vs solubility')

plt.tight_layout()
plt.show()

---
#### **Exercise 1**
Compare the ESOL dataset with Rdkit LogP descriptor

1. Compute the LogP descriptor for each molecule in the ESOL dataset using `Descriptors.MolLogP`.
2. Plot the LogP values against the measured solubility values (logS) in a scatter plot.

In [ ]:
### your code here!

# hint you can use "help(Descriptors)" or help  to see available descriptors in the notebook or check the documentation on rdkit"

---
### A quick tour of RDKit visualization capabilities

In [ ]:
# Visualise a few molecules from each solubility range
df_sorted = df_all.sort_values('logS')
examples = pd.concat([
    df_sorted.head(3),            # poorly soluble
    df_sorted.iloc[len(df_sorted)//2 - 1 : len(df_sorted)//2 + 2],  # medium
    df_sorted.tail(3)             # highly soluble
])

mols  = [Chem.MolFromSmiles(s) for s in examples['smiles']]
labels = [f"logS = {v:.2f}" for v in examples['logS']]
img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 200), legends=labels)
img

---
## 3. Dimensionality Reduction — Building & Understanding the Chemical-Space Toolkit

A **representation** turns a molecule into numbers a computer can work with. Different representations capture different information — so a natural way to compare them is to project each one down to 2D and look at how it arranges the same set of molecules.

We'll build **two reusable tools** and reuse them for every representation in this notebook — including one you design yourself:

1. `plot_chemical_space`: projects any fixed-length representation matrix to 2D — with **PCA**, **t-SNE**, or **UMAP** — (the idea introduced in Lecture 2) and plots it as an **interactive** scatter, colored by solubility, where hovering over a point shows the molecule's structure. PCA axes show the percentage of variance each component explains.
2. `evaluate_representation_knn`: a *quantitative* companion to the plot above. It fits a **k-nearest-neighbor** regressor on a **random** train/test split of the representation and reports the test RMSE/R². A k-NN prediction is only as good as the assumption that "molecules close together in this representation have similar solubility" — so its score is a direct, numeric measure of how well the representation captures the property we care about, independent of any modeling choices that come later in the course. We use a random split here so this check gives a quick, representation-only signal — independent of the single train/test split used later for the Ridge preview (Section 11).

Before turning these tools loose on the real representations later in the notebook, we'll use them on a small hand-picked descriptor set to build intuition for four things every dimensionality-reduction step needs to get right: **standardization**, **removing correlated features**, understanding **what a principal component is actually made of**, and how PCA compares to non-linear methods like **t-SNE**/**UMAP**.

### 3.0 Function helper definitions

In [ ]:
def mol_to_base64_png(mol, size=(150, 100)):
    """Render an RDKit Mol to a base64-encoded PNG data URI, for embedding in a Bokeh tooltip."""
    img = Draw.MolToImage(mol, size=size)
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode('utf-8')

In [ ]:
# Precompute structure images for a fixed subsample once — images only depend on the
# molecule's structure (never on the representation), so every later plot reuses this cache.
N_BOKEH = 1000
bokeh_idx = df_all.sample(N_BOKEH, random_state=42).index

MOL_IMAGES = {}
for i in bokeh_idx:
    mol = Chem.MolFromSmiles(df_all.loc[i, 'smiles'])
    MOL_IMAGES[i] = mol_to_base64_png(mol) if mol else ''

print(f"Cached structure images for {len(MOL_IMAGES)} molecules")

In [ ]:
def project_2d(X, method='pca', n_components=2, random_state=42, standardize=False):
    """Project a representation matrix to `n_components` dimensions.

    Args:
        method: 'pca', 'tsne', or 'umap'.
        standardize: fit a StandardScaler before projecting (default False — see the
            standardization exercise below for why this matters).

    Returns: (coords, info) where info = {
        'explained_variance_ratio': np.ndarray (only for method='pca', else None),
        'reducer': the fitted PCA/TSNE/UMAP object,
    }
    """
    X = np.nan_to_num(np.asarray(X, dtype=float))
    if standardize:
        X = StandardScaler().fit_transform(X)

    if method == 'pca':
        reducer = PCA(n_components=n_components, random_state=random_state)
        coords = reducer.fit_transform(X)
        print(f"Explained variance ratio: {reducer.explained_variance_ratio_.round(3)}")
        info = {'explained_variance_ratio': reducer.explained_variance_ratio_, 'reducer': reducer}
    elif method == 'tsne':
        perplexity = min(30, max(5, len(X) // 100))
        reducer = TSNE(n_components=n_components, random_state=random_state, init='pca', perplexity=perplexity)
        coords = reducer.fit_transform(X)
        info = {'explained_variance_ratio': None, 'reducer': reducer}
    elif method == 'umap':
        reducer = umap.UMAP(n_components=n_components, random_state=random_state)
        coords = reducer.fit_transform(X)
        info = {'explained_variance_ratio': None, 'reducer': reducer}
    else:
        raise ValueError(f"method='{method}' not recognized — use 'pca', 'tsne', or 'umap'")
    return coords, info

In [ ]:
def plot_chemical_space(X, df=None, idx=None, color_col='logS', title='Chemical space',
                         method='pca', standardize=False, n_components=2, random_state=42):
    """Project `X` to 2D and show it as an interactive, hover-to-see-structure Bokeh scatter.

    `X` must have one row per molecule in `df`'s order (default: df_all). Only the
    molecules in `idx` (default: the precomputed bokeh_idx subsample) are drawn, so
    that structure images can be reused from the MOL_IMAGES cache.
    
    `method`/`standardize`/`n_components`/`random_state` are forwarded to `project_2d`.
    For method='pca', the axes are labeled with the percentage of variance each
    component explains; other methods have no such notion, so axes just read 'Dim 1'/'Dim 2'.

    Returns the `info` dict from `project_2d` (e.g. `info['reducer']` for `plot_pca_composition`,
    or `info['explained_variance_ratio']`).
    """
    if df is None:
        df = df_all
    if idx is None:
        idx = bokeh_idx
    if len(X) != len(df):
        raise ValueError(f"X has {len(X)} rows but df has {len(df)} — they must be aligned 1:1")

    coords, info = project_2d(X, method=method, n_components=n_components,
                               random_state=random_state, standardize=standardize)
    sub = df.loc[idx]

    source = ColumnDataSource(data=dict(
        x=coords[idx, 0],
        y=coords[idx, 1],
        smiles=sub['smiles'],
        logS=sub['logS'],
        color_val=sub[color_col],
        image=[MOL_IMAGES[i] for i in idx],
    ))

    mapper = LinearColorMapper(palette=Viridis256, low=sub[color_col].min(), high=sub[color_col].max())

    p = figure(title=title, width=650, height=500, tools='pan,wheel_zoom,reset,save')
    p.circle('x', 'y', source=source, size=8, alpha=0.7,
              fill_color={'field': 'color_val', 'transform': mapper}, line_color=None)
    p.add_layout(ColorBar(color_mapper=mapper, title=color_col), 'right')

    hover = HoverTool(tooltips="""
        <div>
            <img src="@image" style="width:120px;height:80px;"><br>
            <span><b>SMILES:</b> @smiles</span><br>
            <span><b>logS:</b> @logS{0.2f}</span><br>
            <span><b>""" + color_col + """:</b> @color_val{0.2f}</span>
        </div>
    """)
    p.add_tools(hover)

    if method == 'pca' and info['explained_variance_ratio'] is not None:
        evr = info['explained_variance_ratio']
        p.xaxis.axis_label = f'PC1 ({evr[0]*100:.1f}% var)'
        p.yaxis.axis_label = f'PC2 ({evr[1]*100:.1f}% var)'
    else:
        p.xaxis.axis_label = 'Dim 1'
        p.yaxis.axis_label = 'Dim 2'
    show(p)
    return info

In [ ]:
def evaluate_representation_knn(X, y=None, k=5, test_size=0.2, random_state=42):
    """Quantify how well a representation captures solubility structure.

    Fits a k-NN regressor on a RANDOM train/test split of `X` and reports the
    test RMSE/R². Since k-NN predicts using only the labels of nearby points, its
    score is a direct measure of "do nearby points in this representation
    have similar solubility?" — a good proxy for representation quality,
    without needing to train a more complex model.

    Args:
        X: representation matrix, one row per molecule in `df_all`'s order.
        y: target values (default: df_all['logS'].values).
        k: number of neighbors.
        test_size: fraction of molecules held out for testing.

    Returns: (rmse, r2) on the held-out random test split.
    """
    if y is None:
        y = df_all['logS'].values
    X = np.nan_to_num(np.asarray(X, dtype=float))

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state)

    # Scale features so distance is not dominated by whichever column happens
    # to have the largest raw magnitude (e.g. MolWt vs. a 0/1 flag).
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_tr, y_tr)
    y_pred = knn.predict(X_te)

    rmse = root_mean_squared_error(y_te, y_pred)
    r2 = r2_score(y_te, y_pred)
    print(f"k-NN (k={k}, random split) — Test RMSE: {rmse:.3f}  R²: {r2:.3f}")
    return rmse, r2

### 3.1 A small worked example — building intuition before the real representations

To make standardization, correlated features, and PCA composition concrete, we'll compute a handful of RDKit descriptors here — not yet the "official" descriptor set (that comes later, in "Compare with an Expert-Defined Representation", after you've tried building your own representation).

In [ ]:
TEACH_FNS = {
    'MolWt':          Descriptors.MolWt,
    'MolLogP':        Descriptors.MolLogP,
    'TPSA':           Descriptors.TPSA,
    'NumHDonors':     rdMolDescriptors.CalcNumHBD,
    'RingCount':      Descriptors.RingCount,
    'HeavyAtomCount': rdMolDescriptors.CalcNumHeavyAtoms,  # deliberately near-collinear with MolWt
}

def compute_teach_features(smiles_list):
    """Compute a small set of physicochemical descriptors for each SMILES string.
    Args:
        smiles_list: list of SMILES strings.
    Returns:
        pd.DataFrame with one row per SMILES and one column per descriptor in TEACH_FNS.
        If a SMILES string is invalid, the corresponding row is filled with NaNs.
    """
    rows = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rows.append([np.nan] * len(TEACH_FNS))
        else:
            rows.append([fn(mol) for fn in TEACH_FNS.values()])
    return pd.DataFrame(rows, columns=list(TEACH_FNS.keys()))

def plot_pca_composition(pca, feature_names, n_components=2, n_top=10, title_suffix=''):
    """Bar-chart the top contributing features of each principal component.

    `pca`: a fitted sklearn PCA object (e.g. info['reducer'] from project_2d/plot_chemical_space).
    `feature_names`: list of column names, same length as pca.components_.shape[1].

    Only meaningful for named feature blocks (like this teaching set, or the descriptor
    block later) — not for Morgan fingerprint bits or one-hot atom encodings, which have
    no individually interpretable columns.
    """
    feature_names = np.asarray(feature_names)
    fig, axes = plt.subplots(1, n_components, figsize=(6 * n_components, 4))
    if n_components == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        loadings = pca.components_[i]
        top_idx = np.argsort(np.abs(loadings))[::-1][:n_top]
        colors = ['#5DDAB4' if v >= 0 else '#E07070' for v in loadings[top_idx]]
        ax.barh(feature_names[top_idx][::-1], loadings[top_idx][::-1], color=colors[::-1])
        ax.axvline(0, color='white', lw=0.8)
        ax.set_title(f'PC{i+1} composition{title_suffix}')
        ax.set_xlabel('Loading')
    plt.tight_layout()
    plt.show()

In [ ]:
# PCA decomposition of the TEACH physicochemical descriptors

X_teach_all = compute_teach_features(df_all['smiles'])
print(X_teach_all.shape)
X_teach_all.head()

pca  = PCA(n_components=2)
X_pc = pca.fit_transform(X_teach_all)
plot_pca_composition(pca, feature_names=X_teach_all.columns, n_components=2, n_top=6)

In [ ]:
# Visualize the distribution of each descriptor in the TEACH set with a violin plot

import seaborn as sns

plot_df = X_teach_all.loc[:, X_teach_all.notna().any()]  # drop fully-NaN descriptor columns
long_df = plot_df.melt(var_name='Descriptor', value_name='Value').dropna()

plt.figure(figsize=(12, 5))
sns.violinplot(
    data=long_df,
    x='Descriptor',
    y='Value',
    inner='quartile',
    cut=0,
    scale='width',
)
plt.xticks(rotation=45, ha='right')
plt.title('Violin plot of descriptor distributions (train set)')
plt.tight_layout()
plt.show()

### 3.2  Why standardize before PCA?

---
#### **Exercise 2**

**Are the descriptors standardized before PCA is done in 3.1?** Hint check — `project_2d` function and the descriptors value distribution above. Should we standardize before PCA? Why or why not?

Compare PCA on `X_teach_all` **with** and **without** using `project_2d`'s `standardize` option and `plot_pca_composition` How do the PCA axes change? How does the chemical space scatter change? What does this tell you about the importance of standardization?

In [62]:
# your code here!

---
#### **Exercise 3** 
Which single descriptor dominates PC1 in the unstandardized case? Does that match the column with the largest raw numeric range in `X_teach_all.describe()`? 

Going forward, `standardize=True` is the *default* in `project_2d`/`plot_chemical_space` — every later PCA in this notebook is properly scaled.

### 3.3 Removing correlated features

Two columns that measure almost the same thing (e.g. `MolWt` and `HeavyAtomCount` — bigger molecules have both more atoms and more mass) don't add new information; they just double-count it, inflating its apparent importance in PC1. `drop_correlated_features` finds and removes one column from each highly-correlated pair.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(X_teach_all.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature–feature correlation (before removal)')
plt.tight_layout()
plt.show()

In [ ]:
def drop_correlated_features(df, threshold=0.9):
    """Drop one column from each pair of features correlated above `threshold`.

    Returns (reduced_df, dropped_cols, corr_matrix).
    """
    corr = df.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    dropped_cols = [col for col in upper.columns if (upper[col] > threshold).any()]
    reduced_df = df.drop(columns=dropped_cols)
    return reduced_df, dropped_cols, corr

In [ ]:
X_teach_reduced, dropped_cols, corr = drop_correlated_features(X_teach_all, threshold=0.9)
print(f"Dropped columns (|r| > 0.9): {dropped_cols}")
print(f"Shape before: {X_teach_all.shape}  after: {X_teach_reduced.shape}")

_, info_reduced = project_2d(X_teach_reduced.values, standardize=True)
plot_pca_composition(info_reduced['reducer'], X_teach_reduced.columns.tolist())

### 3.4 PCA vs. t-SNE vs. UMAP

PCA is linear — it can miss non-linear cluster structure. `project_2d` also supports `method='tsne'` (scikit-learn, no extra dependency) and `method='umap'` (the `umap-learn` package). Compare all three on the same (correlation-reduced) descriptor block.

In [ ]:
for method in ['pca', 'tsne', 'umap']:
    plot_chemical_space(X_teach_reduced.values, title=f'Teaching descriptors — {method.upper()}', method=method)

---
#### **Exercise 4**
Do t-SNE/UMAP reveal cluster structure PCA misses? Notice their axis labels read "Dim 1"/"Dim 2" rather than a variance-explained percentage — why doesn't "explained variance" make sense for these methods?

**Challenge** — Try t-SNE/UMAP on the *full* descriptor block (with correlated features). How does the chemical space scatter change? What does this tell you about the importance of removing correlated features before non-linear dimensionality reduction?

**Option** - Modify the `plot_chemical_space` function to allow for a `perplexity` argument for t-SNE and a `n_neighbors` argument for UMAP. How do these hyperparameters affect the resulting chemical space scatter? - Alternatively you can take a look at this website for an interactive visualization of the effect of these hyperparameters: 

- t-SNE [https://distill.pub/2016/misread-tsne/](https://distill.pub/2016/misread-tsne/)
- UMAP [https://pair-code.github.io/understanding-umap/](https://pair-code.github.io/understanding-umap/)

In [ ]:
# your code here!

---
## 4. Data Visualization Example

With the toolkit built and the dimensionality-reduction pitfalls understood, here's the toolkit applied end-to-end on a trivial "representation" — just to confirm the pipeline works before using it on anything more interesting.

In [ ]:
# Quick demo: a trivial 2-feature "representation" — molecular weight + heavy-atom count —
# just to check the pipeline works before we use it on anything more interesting.
demo_rows = []
for smi in df_all['smiles']:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        demo_rows.append([Descriptors.MolWt(mol), mol.GetNumHeavyAtoms()])
    else:
        demo_rows.append([np.nan, np.nan])
X_demo = np.array(demo_rows)
X_demo = StandardScaler().fit_transform(X_demo)  # standardize for better visualization (can be done outside of plot_chemical_space)

plot_chemical_space(X_demo, title='Demo: MolWt + heavy-atom count')
evaluate_representation_knn(X_demo)

---
#### **Exercise 5** 
Recolor the plot above by passing `color_col='MolWt'` instead of the default `'logS'`. Does molecular weight explain the same clusters as solubility?

In [ ]:
# your code here!

---
## 5. Build and Evaluate Your Own Representation



---
#### **Exercise 6**

Before looking at what RDKit or the literature suggests, take another look at the molecules you drew earlier (poorly soluble vs. highly soluble). **What do you think drives aqueous solubility?**

Pick 2–4 molecular properties you'd expect to matter — for example: molecule size, polarity, hydrogen-bonding/accepting capacity, aromaticity, or charge....

Then modify `compute_custom_features` (below) to turn each SMILES string into a small vector of your chosen properties, and feed it straight into `plot_chemical_space`. Don't forget to standardize! You don't need RDKit's full descriptor set — simple counts (e.g. number of oxygens, number of rings, number of hydrogen donors, number of rotatable bonds...) are enough to start.

In [ ]:
# your code here!

def compute_custom_features(smiles_list):
    """Your turn: return one feature vector per SMILES. Replace/extend the example below."""
    rows = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None: # avoiding errors for invalid SMILES
            rows.append([np.nan, np.nan]) 
            continue
        # Example starting point — replace with properties YOU think matter:
        desc_1 = np.random.rand() # your first descriptor here
        desc_2 = np.random.rand() # your second descriptor here
        desc_3 = np.random.rand() # your third descriptor here
        # more if you want...
        rows.append([desc_1, desc_2, desc_3]) # don't forget to append your descriptors to the rows list!
    return np.array(rows)

X_custom = compute_custom_features(df_all['smiles'])
plot_chemical_space(X_custom, title='My representation')
evaluate_representation_knn(X_custom)

**Follow-up exercise** — Extend `compute_custom_features` with 2, 10 or 100s more properties and see whether your representation's 2D map — and its `evaluate_representation_knn` score — improves.

In [ ]:
# your code here!

Compare your hypotheses to the expert-crafted descriptor set below — how many did you anticipate? What did you miss? What did you get better?

---
## 6. Compare with an Expert-Defined Representation

### Representation 1 — Physicochemical descriptors

The simplest approach: compute a fixed set of expert-designed features from the 2D structure.

RDKit exposes ~200 descriptors. We will use a curated subset that directly captures solubility-relevant properties:

| Descriptor | Chemical meaning |
|-----------|------------------|
| `MolLogP` | Lipophilicity (Wildman-Crippen) |
| `MolWt` | Molecular weight |
| `NumHDonors` / `NumHAcceptors` | H-bond capacity |
| `TPSA` | Topological polar surface area |
| `NumRotatableBonds` | Flexibility |
| `RingCount` | Aromaticity proxy |

These correspond to the features used in Delaney's original rule-based model.

In [ ]:
DESCRIPTOR_FNS = {
    'MolLogP':           Descriptors.MolLogP,
    'MolWt':             Descriptors.MolWt,
    'NumHDonors':        rdMolDescriptors.CalcNumHBD,
    'NumHAcceptors':     rdMolDescriptors.CalcNumHBA,
    'TPSA':              Descriptors.TPSA,
    'NumRotatableBonds': rdMolDescriptors.CalcNumRotatableBonds,
    'RingCount':         Descriptors.RingCount,
    'FractionCSP3':      rdMolDescriptors.CalcFractionCSP3,
}

def compute_descriptors(smiles_list):
    rows = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rows.append([np.nan] * len(DESCRIPTOR_FNS))
        else:
            rows.append([fn(mol) for fn in DESCRIPTOR_FNS.values()])
    return pd.DataFrame(rows, columns=list(DESCRIPTOR_FNS.keys()))

X_desc_all = compute_descriptors(df_all['smiles'])

print(X_desc_all.shape)
X_desc_all.head()

### Representation 1b — Charge & protonation state

Solubility in water is strongly affected by whether a molecule is charged or has groups that can gain/lose a proton at physiological pH. We add two kinds of features:

- **Formal charge** and a coarse **ionizable-group count** — computed directly from the SMILES with RDKit (a few SMARTS patterns for carboxylic acids, amines, phenols, anilines). This is a heuristic, not a real pKa prediction.
- **pKa**, joined in from a precomputed dataset (`tutorials/data/esol_pka.csv`) keyed by canonical SMILES. **This file currently ships with only a handful of illustrative rows — replace it with the instructor's real pKa predictions before class.** Molecules without a match keep `pka` as missing (`has_pka_data = 0`), imputed with the training-set median only when building the model input (no test-set leakage).

In [ ]:
# --- Charge & ionizable-group heuristic (RDKit only) ---
IONIZABLE_PATTERNS = {
    'carboxylic_acid': Chem.MolFromSmarts('C(=O)[OH]'),
    'phenol':          Chem.MolFromSmarts('c[OH]'),
    'primary_amine':   Chem.MolFromSmarts('[NX3;H2;!$(NC=O)]'),
    'secondary_amine': Chem.MolFromSmarts('[NX3;H1;!$(NC=O)]'),
    'aniline':         Chem.MolFromSmarts('c[NX3]'),
}

def canonical_smiles(smi):
    mol = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(mol) if mol else None

def charge_features(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return {'FormalCharge': np.nan, 'NumIonizableGroups': np.nan}
    n_ionizable = sum(len(mol.GetSubstructMatches(patt)) for patt in IONIZABLE_PATTERNS.values())
    return {'FormalCharge': float(Chem.GetFormalCharge(mol)), 'NumIonizableGroups': float(n_ionizable)}

def add_charge_features(df):
    feats = pd.DataFrame([charge_features(s) for s in df['smiles']], index=df.index)
    return pd.concat([df, feats], axis=1)

# --- pKa enrichment (instructor-supplied CSV, keyed by canonical SMILES) ---
pka_df = pd.read_csv('data/esol_pka.csv')
pka_df['smiles_canonical'] = pka_df['smiles'].apply(canonical_smiles)
pka_df = pka_df.drop_duplicates(subset='smiles_canonical')

def merge_pka(df):
    df = df.copy()
    df['smiles_canonical'] = df['smiles'].apply(canonical_smiles)
    merged = df.merge(pka_df[['smiles_canonical', 'pka', 'is_ionizable']], on='smiles_canonical', how='left')
    n_matched = int(merged['pka'].notna().sum())
    print(f"pKa merge: {n_matched}/{len(merged)} matched, {len(merged) - n_matched} unmatched (kept as NaN)")
    merged['has_pka_data'] = merged['pka'].notna().astype(int)
    merged['is_ionizable'] = merged['is_ionizable'].fillna(0).astype(int)
    return merged.drop(columns=['smiles_canonical'])

df_all = add_charge_features(merge_pka(df_all))

# Model-input block: same numeric columns, pKa imputed with the dataset median
pka_median = df_all['pka'].median()

def build_charge_block(df):
    block = df[['FormalCharge', 'NumIonizableGroups', 'has_pka_data']].reset_index(drop=True).copy()
    block['pka'] = df['pka'].fillna(pka_median).reset_index(drop=True)
    return block

X_desc_all = pd.concat([X_desc_all, build_charge_block(df_all)], axis=1)

print(X_desc_all.shape)
X_desc_all.head()

In [ ]:
# Correlation of each descriptor with logS
# Note: this is correlation with logS (the target), not feature-feature correlation —
# see Section 3.3 (drop_correlated_features) for that.
corr_df = X_desc_all.copy()
corr_df['logS'] = df_all['logS'].values
corr = corr_df.corr()['logS'].drop('logS').sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#E07070' if v < 0 else '#5DDAB4' for v in corr]
ax.barh(corr.index, corr.values, color=colors, edgecolor='none')
ax.axvline(0, color='white', lw=0.8)
ax.set_xlabel('Pearson r with logS')
ax.set_title('Descriptor correlations with solubility')
plt.tight_layout()
plt.show()

In [ ]:
X_desc_all = X_desc_all.values
X_desc_all = StandardScaler().fit_transform(X_desc_all)  # standardize for better visualization (can be done outside of plot_chemical_space)
plot_chemical_space(X_desc_all, title='Descriptor space (incl. charge/pKa)')
evaluate_representation_knn(X_desc_all)

---
## 7. Morgan Fingerprints (ECFP4)

Morgan fingerprints encode the **local chemical environment** of each atom out to a given radius. With radius=2 and 2048 bits they are known as **ECFP4**.

```
Molecule → iterate atom environments (radius 0,1,2) → hash → fold to 2048-bit vector
```

Key properties:
- **Fixed-length** bit vector (or count vector)
- **No explicit bond-order/chirality** by default (configurable)
- **Sparse** — typically <5% of bits set
- Highly effective for similarity search and ML baselines

Below, we compute the fingerprints, highlight how sparse they are, then clean out zero-variance bits before visualizing.

In [ ]:
def morgan_matrix(smiles_list, radius=2, n_bits=2048):
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            fp = generator.GetFingerprint(mol)
            arr = np.zeros(n_bits, dtype=np.uint8)
            DataStructs.ConvertToNumpyArray(fp, arr)
            fps.append(arr)
        else:
            fps.append(np.zeros(n_bits, dtype=np.uint8))
    return np.array(fps)

X_fp_all = morgan_matrix(df_all['smiles'])

print(f"Fingerprint matrix shape: {X_fp_all.shape}")
print(f"Average bit density: {X_fp_all.mean()*100:.1f}%")

---
#### **Exercise 7**

1. Change `radius` from 2 to 3 (ECFP6) in the call above and rerun the fingerprint-space plot further down. How does the 2D map shift?

2. Explore the RDKit FingerprintGenerator API to see what other options are available beyond Morgan fingerprints (e.g. MACCS, AtomPairs, TopologicalTorsions) and try one of them instead. Hint: the blog post [RDKit FingerprintGenerator](https://greglandrum.github.io/rdkit-blog/posts/2023-01-18-fingerprint-generator-tutorial.html) is a good starting point.

3. Using the morgan fingerprint representation, find the bit that is set for the most molecules. What substructure does it correspond to? (Hint: use `rdkit.Chem.rdMolDescriptors.GetMorganFingerprintAsBitVect` with `bitInfo` to get the atom environments that set each bit.)


In [ ]:
# your code here!

### Visualizing the fingerprint space

In [ ]:
# Visualise the fingerprint sparsity for the first 20 molecules
fig, ax = plt.subplots(figsize=(12, 3))
ax.imshow(X_fp_all[:20], aspect='auto', cmap='Greens', interpolation='nearest')
ax.set_xlabel('Bit index (2048)')
ax.set_ylabel('Molecule')
ax.set_title('ECFP4 fingerprints — first 20 molecules (green = bit set)')
plt.tight_layout()
plt.show()

# How often is each bit set across the whole dataset?
bit_density = X_fp_all.mean(axis=0)
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(bit_density, bins=50, color='#5DDAB4')
ax.set_xlabel('Fraction of molecules with bit set')
ax.set_ylabel('Number of bits')
ax.set_title('Per-bit sparsity across the fingerprint')
plt.tight_layout()
plt.show()

### Removing zero-variance bits

As illustrated above, many of the 2048 bits never fire on this ~1128-molecule dataset — they carry no information and only add noise to distance calculations (like the k-NN check below). `sklearn.feature_selection.VarianceThreshold` removes them.

In [ ]:
vt = VarianceThreshold(threshold=0.0)  # drops any bit that's constant across every molecule, you can try to remove bits with low variance by setting a threshold > 0.0
X_fp_all_reduced = vt.fit_transform(X_fp_all)
X_fp_all_reduced = StandardScaler().fit_transform(X_fp_all_reduced)  

print(f"Fingerprint shape before: {X_fp_all.shape}  after removing zero-variance bits: {X_fp_all_reduced.shape}")

In [ ]:
plot_chemical_space(X_fp_all_reduced, title='Fingerprint space (ECFP4, zero-variance bits removed)')
evaluate_representation_knn(X_fp_all_reduced)

---
#### **Exercise 8** 
Compute a **Tanimoto similarity matrix** between the 30 least-soluble molecules and the 30 most-soluble ones using ECFP4 (`DataStructs.TanimotoSimilarity` or `BulkTanimotoSimilarity`). Visualise it as a heatmap. What does it tell you about the chemical space of solubility?

In [ ]:
# your code here!

---
## 8. Featurizing with xtb/APLB for solvation energy

So far every representation has come from 2D structure only. Quantum-chemistry tools like **xtb** (a fast semi-empirical tight-binding method) let us compute an actual energy from a 3D conformer — including a solvation correction. The idea: the change in energy between vacuum and water (ΔG_solv) is itself a solubility-relevant descriptor — a charged/polar molecule should be stabilized much more by water than a nonpolar one.

We'll (a) compute it live with xtb on a small subsample to see how it works and how slow it is, then (b) load a precomputed value for the full dataset, using the same enrichment pattern as the pKa data above: a CSV keyed by canonical SMILES, merged in with unmatched rows kept as `NaN` rather than dropped.

In [ ]:
def embed_3d(mol, random_seed=42, xtb_preopt=True):
    """Add explicit hydrogens and generate a 3D conformer (ETKDG embedding + a quick
    force-field pre-optimization) so xtb has coordinates to work with."""
    mol = Chem.AddHs(mol)
    ok = AllChem.EmbedMolecule(mol, randomSeed=random_seed, useRandomCoords=True)
    if ok != 0:
        # ETKDG can occasionally fail to find an embedding — retry once with a different seed.
        ok = AllChem.EmbedMolecule(mol, randomSeed=random_seed + 1, useRandomCoords=True)
        if ok != 0:
            return None
    try:
        AllChem.MMFFOptimizeMolecule(mol)
    except ValueError:
        AllChem.UFFOptimizeMolecule(mol)  # fall back if MMFF has no parameters for this molecule

    if xtb_preopt:
        # xtb can be sensitive to the initial geometry, so do a quick xtb pre-optimization
        # (GFN2-xTB single-point energy + geometry optimization) before the final energy call.
        with tempfile.TemporaryDirectory() as tmpdir:
            xyz_path = f'{tmpdir}/mol.xyz'
            with open(xyz_path, 'w') as f:
                xyz_block = Chem.MolToXYZBlock(mol)
                f.write(xyz_block)
            charge = Chem.GetFormalCharge(mol)
            cmd = ['xtb', 'mol.xyz', '--gfn', '2', '-c', str(charge), '--opt']
            subprocess.run(cmd, capture_output=True, text=True, cwd=tmpdir)
            mol = Chem.MolFromXYZFile(f'{tmpdir}/xtbopt.xyz')
            
    return mol

def run_xtb_energy(xyz_block, charge=0, solvent=None):
    """Run a single xtb GFN2-xTB single-point calculation and return the total energy
    (Hartree). `solvent`, if given (e.g. 'water'), adds an ALPB implicit-solvation correction."""
    with tempfile.TemporaryDirectory() as tmpdir:
        xyz_path = f'{tmpdir}/mol.xyz'
        with open(xyz_path, 'w') as f:
            f.write(xyz_block)

        cmd = ['xtb', 'mol.xyz', '--gfn', '2', '-c', str(charge)]
        if solvent:
            cmd += ['--alpb', solvent]
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=tmpdir)

        for line in result.stdout.splitlines():
            if 'TOTAL ENERGY' in line:
                return float(line.split()[3])
        raise RuntimeError(f"Could not find TOTAL ENERGY in xtb output:\n{result.stdout[-1000:]}")

def compute_solvation_energy(smi):
    """Gas-phase vs. ALPB-water xtb single-point energy for one molecule.
    Returns None if 3D embedding fails."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    mol_3d = embed_3d(mol)
    if mol_3d is None:
        return None
    xyz = Chem.MolToXYZBlock(mol_3d)
    charge = Chem.GetFormalCharge(mol)
    e_gas = run_xtb_energy(xyz, charge=charge)
    e_solv = run_xtb_energy(xyz, charge=charge, solvent='water')
    return {
        'E_gas_hartree': e_gas,
        'E_solv_hartree': e_solv,
        'dG_solv_kcal': (e_solv - e_gas) * 627.509,  # Hartree -> kcal/mol
    }

In [ ]:
# Live demo: run xtb on a small subsample to see it work — and see why it doesn't scale.
from tqdm.notebook import tqdm
xtb_sample = df_all.sample(1128, random_state=42)

# try loading the xtb results first to avoid re-running the calculations if they were already done
xtb_results_path = 'data/esol_xtb_solvation.csv'
df_xtb_results = pd.read_csv(xtb_results_path)
computed_smiles = set(df_xtb_results['smiles'])

t0 = time.time()
xtb_results = {}
for smi in tqdm(xtb_sample['smiles']):
    if smi not in computed_smiles:
        result = compute_solvation_energy(smi)
        if result is not None:
            xtb_results[smi] = result
    else:
        # reuse the previously computed result
        row = df_xtb_results[df_xtb_results['smiles'] == smi].iloc[0]
        xtb_results[smi] = {
            'E_gas_hartree': row['E_gas_hartree'],
            'E_solv_hartree': row['E_solv_hartree'],
            'dG_solv_kcal': row['dG_solv_kcal'],
        }
elapsed = time.time() - t0

print(f"{len(xtb_results)}/{len(xtb_sample)} molecules succeeded in {elapsed:.1f}s ({elapsed/len(xtb_sample):.2f}s/molecule)")
print(f"Extrapolated to the full {len(df_all)}-molecule dataset: ~{elapsed/len(xtb_sample)*len(df_all)/60:.1f} minutes — too slow for a live tutorial.")

In [ ]:
df_xtb_results = pd.DataFrame.from_dict(xtb_results, orient='index').reset_index().rename(columns={'index': 'smiles'})
df_xtb_results.to_csv(xtb_results_path, index=False)

### Using precomputed values for the full dataset

`tutorials/data/esol_xtb_solvation.csv` (same convention as `esol_pka.csv`: keyed by canonical SMILES, a small set of illustrative rows — **replace with real xtb output before class**) is merged in exactly like the pKa data: left-merge on canonical SMILES, train-median imputation, `has_xtb_data` flag, no rows dropped.

In [ ]:
xtb_df = pd.read_csv('data/esol_xtb_solvation.csv')
xtb_df['smiles_canonical'] = xtb_df['smiles'].apply(canonical_smiles)
xtb_df = xtb_df.drop_duplicates(subset='smiles_canonical')

def merge_xtb(df, xtb_df):
    df = df.copy()
    df['smiles_canonical'] = df['smiles'].apply(canonical_smiles)
    merged = df.merge(xtb_df[['smiles_canonical', 'dG_solv_kcal']], on='smiles_canonical', how='left')
    n_matched = int(merged['dG_solv_kcal'].notna().sum())
    print(f"xtb merge: {n_matched}/{len(merged)} matched, {len(merged) - n_matched} unmatched (kept as NaN)")
    merged['has_xtb_data'] = merged['dG_solv_kcal'].notna().astype(int)
    return merged.drop(columns=['smiles_canonical'])

df_xtb = merge_xtb(df_all, xtb_df)


X_xtb_all = df_xtb[['dG_solv_kcal', 'has_xtb_data']].copy()
X_xtb_all = VarianceThreshold(threshold=0.0).fit_transform(X_xtb_all)  # drops any feature that's constant across every molecule
X_xtb_all = StandardScaler().fit_transform(X_xtb_all)  
X_xtb_all.shape

print(X_xtb_all.shape)

In [ ]:
# Visualize the relationship between xtb solvation energy and experimental solubility
plt.scatter(df_xtb['dG_solv_kcal'], df_xtb['logS'], alpha=0.4, s=12, c='#5DDAB4')
plt.xlabel('xtb solvation energy (kcal/mol)')
plt.ylabel('log(S) [mol/L]')
plt.title('xtb solvation energy vs. experimental solubility')
plt.tight_layout()
plt.show()

**Note:** Other MLIPs such as Aimnet or Meta's "UMA-s" universal ML interatomic potential could replace xtb here for a better energy estimate.

---
## 9. Featurizing with a Foundation Model (ChemBERTa)

`seyonec/ChemBERTa-zinc-base-v1` (the same checkpoint used in Tutorial 9 / Lecture 7) is a BERT-style model pretrained on ~77M SMILES from ZINC. Without any fine-tuning, its internal representations already encode useful chemical structure — we extract a frozen embedding per molecule and treat it exactly like any other representation in this notebook.

In [ ]:
CHEMBERTA_MODEL = 'seyonec/ChemBERTa-zinc-base-v1'
_chemberta_tokenizer = AutoTokenizer.from_pretrained(CHEMBERTA_MODEL)
_chemberta_model = AutoModel.from_pretrained(CHEMBERTA_MODEL)

_chemberta_device = 'cuda' if torch.cuda.is_available() else 'cpu'
_chemberta_model.to(_chemberta_device).eval()

print(f"Loaded {CHEMBERTA_MODEL} on {_chemberta_device}")

In [ ]:
def chemberta_embed(smiles_list, batch_size=32):
    """Frozen ChemBERTa CLS-token embedding, one 768-dim vector per SMILES."""
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(smiles_list), batch_size):
            batch = list(smiles_list[i:i + batch_size])
            enc = _chemberta_tokenizer(batch, padding=True, truncation=True, return_tensors='pt')
            enc = {k: v.to(_chemberta_device) for k, v in enc.items()}
            out = _chemberta_model(**enc)
            cls = out.last_hidden_state[:, 0, :]  # CLS token
            embeddings.append(cls.cpu().numpy())
    return np.vstack(embeddings)

In [ ]:
X_chemberta_all = chemberta_embed(df_all['smiles'].tolist())
print(f"ChemBERTa embedding shape: {X_chemberta_all.shape}")
X_chemberta_all = StandardScaler().fit_transform(X_chemberta_all)  

plot_chemical_space(X_chemberta_all, title='ChemBERTa embedding space')
evaluate_representation_knn(X_chemberta_all)

**Caveat:** this requires internet access at runtime to download the ~350MB checkpoint the first time (a Colab/local network caveat, similar to xtb's binary-installation caveat above).

---
## 10. Graph Definition

Instead of hashing atom environments into a fixed vector, we represent the molecule **as-is**: atoms are nodes, bonds are edges.

```
Molecule → atoms (node features) + bonds (edge indices + edge features) → Graph
```

### Node features (per atom)
| Feature | Values |
|---------|--------|
| Atomic number | one-hot (C, N, O, S, F, Cl, Br, other) |
| Degree | 0–5 |
| Formal charge | int |
| Hybridization | SP, SP2, SP3 |
| Aromaticity | 0/1 |
| Num H | 0–4 |

### Edge features (per bond)
| Feature | Values |
|---------|--------|
| Bond type | one-hot (single, double, triple, aromatic) |

Unlike the previous representations, a graph has no fixed-size vector — the number of nodes and edges varies per molecule — so it doesn't fit into `plot_chemical_space` (PCA needs equal-length rows). We build the representation here and stop: **training a model that consumes it — a GNN with graph convolutions and pooling — is covered in Lecture 5 (GNNs/MLIPs)**. Come back to `graph_all` then.

> **Key difference from fingerprints:** a GNN *learns* which structural patterns matter for the property, rather than using predetermined hash buckets — but it needs to be trained to do that.

In [ ]:
ATOM_TYPES = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'I', 'P', 'other']
BOND_TYPES = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
              Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
EDGE_DIM = len(BOND_TYPES)

def atom_features(atom):
    """Return a 1D feature vector for a single atom."""
    # Atom type one-hot
    sym = atom.GetSymbol()
    atom_oh = [int(sym == t) for t in ATOM_TYPES[:-1]] + [int(sym not in ATOM_TYPES[:-1])]
    # Degree (capped at 5)
    degree_oh = [int(atom.GetDegree() == d) for d in range(6)]
    # Hybridization
    hyb = atom.GetHybridization()
    hyb_oh = [
        int(hyb == Chem.rdchem.HybridizationType.SP),
        int(hyb == Chem.rdchem.HybridizationType.SP2),
        int(hyb == Chem.rdchem.HybridizationType.SP3),
    ]
    # Scalar features
    scalars = [
        atom.GetFormalCharge(),
        int(atom.GetIsAromatic()),
        atom.GetTotalNumHs(),
    ]
    return atom_oh + degree_oh + hyb_oh + scalars

def bond_features(bond):
    """One-hot bond type: single, double, triple, aromatic."""
    bt = bond.GetBondType()
    return [int(bt == t) for t in BOND_TYPES]

NODE_DIM = len(atom_features(Chem.MolFromSmiles('C').GetAtomWithIdx(0)))
print(f"Node feature dimension: {NODE_DIM}")
print(f"Edge feature dimension: {EDGE_DIM}")

def smiles_to_graph(smi, y_val):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    # Node features
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    # Edge indices + edge features (undirected → add both directions)
    edges = []
    edge_attrs = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edges += [[i, j], [j, i]]
        edge_attrs += [bf, bf]
    if edges:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, EDGE_DIM), dtype=torch.float)
    y = torch.tensor([y_val], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

# Build the graph dataset
graph_all = [g for g in (smiles_to_graph(s, y) for s, y in zip(df_all['smiles'], df_all['logS'])) if g]

print(f"Graph dataset: {len(graph_all)} molecules")
print(f"Example graph: {graph_all[0]}")

In [ ]:
# Compare two very different molecules' graphs directly: a small one vs. the largest in the dataset.
small_smiles = 'CCO'  # ethanol
big_idx = df_all['smiles'].apply(lambda s: Chem.MolFromSmiles(s).GetNumHeavyAtoms() if Chem.MolFromSmiles(s) else 0).idxmax()
big_smiles = df_all.loc[big_idx, 'smiles']

g_small = smiles_to_graph(small_smiles, 0.0)
g_big = smiles_to_graph(big_smiles, 0.0)

print(f"Ethanol ({small_smiles}):")
print(f"  x.shape={tuple(g_small.x.shape)}  edge_index.shape={tuple(g_small.edge_index.shape)}  edge_attr.shape={tuple(g_small.edge_attr.shape)}")
print(f"Largest molecule ({big_smiles}):")
print(f"  x.shape={tuple(g_big.x.shape)}  edge_index.shape={tuple(g_big.edge_index.shape)}  edge_attr.shape={tuple(g_big.edge_attr.shape)}")

### Why graphs need pooling for visualization

The two molecules above have different `x.shape` — a graph has no fixed-size vector, so `plot_chemical_space`/PCA can't consume it directly, as noted above. A GNN solves this by learning a **pooling** operation that aggregates all of a molecule's (variable-count) node features into one fixed-size vector.

As a preview — without training an actual GNN (deferred to Lecture 5) — we can do the simplest possible version of this by hand: mean-pool the raw atom features.

In [ ]:
def graph_mean_pool(graph_list):
    """Mean atom-feature vector per molecule — a hand-rolled preview of GNN pooling."""
    return np.array([g.x.numpy().mean(axis=0) for g in graph_list])

In [ ]:
# Visualize what graph_mean_pool actually does: N per-atom feature rows collapse into 1 averaged row.
mol_small = Chem.MolFromSmiles(small_smiles)
atom_labels = [a.GetSymbol() for a in mol_small.GetAtoms()]
node_matrix = g_small.x.numpy()
pooled_vector = node_matrix.mean(axis=0, keepdims=True)

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(10, 4), sharex=True,
    gridspec_kw={'height_ratios': [len(atom_labels), 1]}
)
vmax = max(node_matrix.max(), pooled_vector.max())
ax1.imshow(node_matrix, aspect='auto', cmap='viridis', vmin=0, vmax=vmax)
ax1.set_yticks(range(len(atom_labels)))
ax1.set_yticklabels(atom_labels)
ax1.set_title(f'Per-atom feature vectors — {small_smiles} ({len(atom_labels)} atoms × {NODE_DIM} features)')

ax2.imshow(pooled_vector, aspect='auto', cmap='viridis', vmin=0, vmax=vmax)
ax2.set_yticks([0])
ax2.set_yticklabels(['mean'])
ax2.set_xlabel('Feature index')
ax2.set_title(f'graph_mean_pool output — one 1×{NODE_DIM} vector (averaged column-wise)')

plt.tight_layout()
plt.show()

In [ ]:
X_graphmean_all = graph_mean_pool(graph_all)
print(f"Mean-pooled graph representation shape: {X_graphmean_all.shape}")

plot_chemical_space(X_graphmean_all, title='Mean-pooled atom features (preview of GNN pooling)')
evaluate_representation_knn(X_graphmean_all)

---
#### **Exercise 9**
Implement a simple mean-pooling function that concatenates the mean of node features and the mean of edge features for each molecule. Use it to turn `graph_all` into a fixed-size representation matrix and visualize it with `plot_chemical_space`. How does the chemical space scatter compare to the node only information?

In [ ]:
# your code here!

---
## 11. Preview — From Representation to Prediction

So far we've only built and visualized representations — no model has seen the solubility labels yet. As a small preview of what's coming in later lectures (Supervised Learning, Neural Networks, GNNs), let's fit the simplest possible model, **Ridge regression**, on the descriptor representation (including the charge/pKa features) we built above.

In [ ]:
# The one place in this notebook we actually need a train/test split: a simple random
# split, introduced here rather than threaded through every representation above.
idx_train, idx_test = train_test_split(np.arange(len(df_all)), test_size=0.2, random_state=42)

ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Ridge(alpha=1.0))
])
ridge.fit(X_desc_all[idx_train], df_all['logS'].values[idx_train])

y_pred_ridge = ridge.predict(X_desc_all[idx_test])
rmse_ridge = root_mean_squared_error(df_all['logS'].values[idx_test], y_pred_ridge)
r2_ridge   = r2_score(df_all['logS'].values[idx_test], y_pred_ridge)

print(f"Ridge regression (descriptors) — Test RMSE: {rmse_ridge:.3f}  R²: {r2_ridge:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
y_true = df_all['logS'].values[idx_test]
ax.scatter(y_true, y_pred_ridge, alpha=0.5, s=18, c='#5DDAB4')
lim = [min(y_true.min(), y_pred_ridge.min()) - 0.3, max(y_true.max(), y_pred_ridge.max()) + 0.3]
ax.plot(lim, lim, '--', lw=1)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Measured logS')
ax.set_ylabel('Predicted logS')
ax.set_title('Ridge (descriptors + charge/pKa) — test set')
ax.text(lim[0] + 0.2, lim[1] - 0.2, f"RMSE: {rmse_ridge:.3f}\nR²: {r2_ridge:.3f}", fontsize=10, ha='left', va='top', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
plt.tight_layout()
plt.show()

---
#### **Exercise 10**

Iterate over the representations you built above (your own, descriptors, fingerprints, ChemBERTa, and the graph mean-pooled representation) and compare their Ridge regression performance. Which representation gives the best RMSE/R²? How does that compare to your expectations from the chemical-space visualizations?


In [ ]:
# your code here!

---
## 12. Discussion

### Discussion

| Representation | Strengths | Limitations |
|---------------|-----------|-------------|
| Physicochemical descriptors (+ charge/pKa) | Interpretable, fast, domain-informed | Fixed feature set; misses structural patterns not encoded |
| Morgan fingerprints | Captures local environments, widely validated | Bit collisions; no spatial awareness; fixed radius |
| Molecular graph | Flexible, no information loss — atoms and bonds kept as-is | No fixed-size vector — needs a model like a GNN to consume it directly (Lecture 5) |

**Note on the Ridge split:** for simplicity, every representation above is built on the full dataset, and we only split into train/test once — a random split, right before fitting Ridge. A *scaffold* split (Bemis-Murcko), where the test molecules have chemical scaffolds unseen during training, is a harder and more realistic test, and typically gives worse (higher) errors than the random split used here — worth trying once train/test splitting is covered in more depth later in the course.

---
## 13. Further Practice

Most exercises now live right next to the representation they belong to (standardization in Section 3, ECFP6 radius in Section 7, extending your custom representation in Section 5, etc.). The ones below need *every* representation to already exist, so they live here instead:

1. Rank the demo, descriptor, fingerprint, xtb, ChemBERTa, and mean-pooled-graph representations by their `evaluate_representation_knn` RMSE. Does the ranking match your visual impression from the Bokeh plots? Try a few different values of `k` — does the ranking change?
2. Compare `evaluate_representation_knn` RMSE for the mean-pooled graph representation with vs. without folding bond-order information into a per-node aggregate (e.g. append each atom's mean incident bond order, computed from `edge_attr`, as an extra column before pooling). Does edge information change the pooled result?
3. Concatenate the descriptor block, the xtb `dG_solv_kcal` column, and the ChemBERTa embedding into one combined representation. Does `evaluate_representation_knn` improve over any single representation alone?

In [ ]:
# Your code here


---
## Further Reading

- **Rogers & Hahn (2010)** — *Extended-Connectivity Fingerprints* — J. Chem. Inf. Model. 50, 742 — the ECFP reference
- **Duvenaud et al. (2015)** — *Convolutional Networks on Graphs for Learning Molecular Fingerprints* — NeurIPS
- **Yang et al. (2019)** — *Analyzing Learned Molecular Representations for Property Prediction* — J. Chem. Inf. Model. 59, 3370 (Chemprop paper — benchmarks fingerprints vs graphs on MoleculeNet)
- **David et al. (2020)** — *Molecular Representations in AI-driven Drug Discovery* — J. Cheminform. 12, 56 — accessible review
- **Delaney (2004)** — *ESOL: Estimating Aqueous Solubility Directly from Molecular Structure* — J. Chem. Inf. Comput. Sci. 44, 1000 — original dataset paper
- **Pan et al. (2021)** — *MolGpKa: A Web Server for Small Molecule pKa Prediction Using a Graph-Convolutional Neural Network* — J. Chem. Inf. Model. 61, 3159 — an example pKa-prediction approach for enriching `esol_pka.csv`
- **Bannwarth, Ehlert & Grimme (2019)** — *GFN2-xTB* — J. Chem. Theory Comput. 15, 1652 — the xtb method used in Section 8
- **Ehlert et al. (2021)** — *Robust and Efficient Implicit Solvation Model for Fast Semiempirical Methods (ALPB)* — J. Chem. Theory Comput. 17, 4250 — the water solvation model used in Section 8
- **Chithrananda, Grand & Ramsundar (2020)** — *ChemBERTa: Large-Scale Self-Supervised Pretraining for Molecular Property Prediction* — arXiv:2010.09885 — the foundation model used in Section 9
- **van der Maaten & Hinton (2008)** — *Visualizing Data using t-SNE* — J. Mach. Learn. Res. 9, 2579
- **McInnes, Healy & Melville (2018)** — *UMAP: Uniform Manifold Approximation and Projection* — arXiv:1802.03426
- **Bokeh documentation** — [`HoverTool`](https://docs.bokeh.org/en/latest/docs/user_guide/interaction/tooltips.html) and [`ColumnDataSource`](https://docs.bokeh.org/en/latest/docs/user_guide/basic/data.html) — the two APIs behind `plot_chemical_space`